# Step 1.2 — Extract the RAND HRS Codebook

**Purpose:** Extract searchable text from the RAND HRS Codebook PDF, preserving page numbers and recording extraction quality indicators.

**Outputs (Delta tables):**
- `dev_catalog.hrs_agent_metadata.hrs_document_pages` — one record per PDF page.
- `dev_catalog.hrs_agent_metadata.hrs_document_processing_runs` — one record per extraction run.

**Before running:** Update `PDF_PATH` in the configuration cell to the actual PDF path in a Unity Catalog Volume or another path accessible to this notebook.

This notebook creates a searchable extraction foundation. It does **not** claim that every variable definition or descriptive-statistics table has been perfectly structured. Later steps should parse and verify those records against the original PDF pages.

In [ ]:
# Cell 1 — Configuration
# Change PDF_PATH to the actual uploaded Codebook location.
# Recommended: a Unity Catalog Volume path such as:
# /Volumes/<catalog>/<schema>/<volume>/<filename>.pdf

PDF_PATH = "/Volumes/landing_catalog/external_data/rand_hrs_codebook/rand_hrs_codebook.pdf"

TARGET_CATALOG = "dev_catalog"
TARGET_SCHEMA = "hrs_agent_metadata"

PAGES_TABLE = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.hrs_document_pages"
RUNS_TABLE = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.hrs_document_processing_runs"

# Optional descriptive metadata. Update these values for the exact PDF edition.
DOCUMENT_ID = "rand_hrs_longitudinal_codebook"
DOCUMENT_TITLE = "RAND HRS Longitudinal Codebook"
CODEBOOK_VERSION = "unknown"  # e.g., edition/year, once verified

# If True, reprocess all pages and replace the existing page table contents.
# If False, this notebook still extracts the PDF but appends the run's pages.
REPLACE_PAGE_TABLE = True

print(f"PDF path: {PDF_PATH}")
print(f"Output pages table: {PAGES_TABLE}")
print(f"Output runs table: {RUNS_TABLE}")

In [ ]:
# Cell 2 — Confirm source file and install/import PDF extraction library
# PyMuPDF provides page-level text extraction. If the package is not available
# in the serverless environment, this cell attempts to install it for the session.

import os
import sys
import subprocess

if not os.path.isfile(PDF_PATH):
    raise FileNotFoundError(
        f"PDF not found at {PDF_PATH!r}. Update PDF_PATH to the actual file path. "
        "For Unity Catalog Volumes, use /Volumes/<catalog>/<schema>/<volume>/<file>.pdf."
    )

try:
    import fitz  # PyMuPDF
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymupdf"])
    import fitz

file_size_bytes = os.path.getsize(PDF_PATH)
print(f"Found PDF ({file_size_bytes:,} bytes). PyMuPDF version: {fitz.VersionBind}")

In [ ]:
# Cell 3 — Create output schema and processing-run table

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {RUNS_TABLE} (
    run_id STRING,
    document_id STRING,
    document_title STRING,
    codebook_version STRING,
    source_path STRING,
    source_file_size_bytes BIGINT,
    source_page_count INT,
    extracted_page_count INT,
    empty_or_low_text_page_count INT,
    extraction_status STRING,
    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    notes STRING
)
USING DELTA
''')

print(f"Metadata schema and run table are ready: {TARGET_CATALOG}.{TARGET_SCHEMA}")

In [ ]:
# Cell 4 — Extract page text and assign review flags

from datetime import datetime, timezone
import hashlib
import re

run_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc).replace(tzinfo=None)

with fitz.open(PDF_PATH) as pdf:
    source_page_count = pdf.page_count
    extracted_rows = []

    for page_index in range(source_page_count):
        page = pdf.load_page(page_index)
        raw_text = page.get_text("text") or ""
        # Normalize line endings and remove trailing whitespace while retaining
        # meaningful line boundaries for later variable/table parsing.
        page_text = "\n".join(line.rstrip() for line in raw_text.replace("\r\n", "\n").replace("\r", "\n").split("\n")).strip()

        # Heuristic indicators only; these are not definitive classifications.
        upper_text = page_text.upper()
        variable_tokens = sorted(set(re.findall(r"\b[A-Z][A-Z0-9]{1,11}\b", upper_text)))
        mentions_statistics = bool(re.search(
            r"\b(MEAN|MEDIAN|MINIMUM|MAXIMUM|STD\.? DEV|STANDARD DEVIATION|PERCENTILE|FREQUENCY|DISTRIBUTION)\b",
            upper_text
        ))
        mentions_variable = bool(re.search(
            r"\b(VARIABLE NAME|VARIABLE LABEL|VARIABLE DESCRIPTION|VARIABLE|R\d+[A-Z0-9_]*|HHIDPN)\b",
            upper_text
        ))

        # Mark pages that likely need OCR or visual review.
        text_length = len(page_text)
        if text_length < 40:
            review_flag = "REVIEW_LOW_TEXT"
        elif "\ufffd" in page_text:
            review_flag = "REVIEW_ENCODING"
        else:
            review_flag = "NONE"

        text_hash = hashlib.sha256(page_text.encode("utf-8")).hexdigest()

        extracted_rows.append({
            "run_id": run_id,
            "document_id": DOCUMENT_ID,
            "document_title": DOCUMENT_TITLE,
            "codebook_version": CODEBOOK_VERSION,
            "source_file_name": os.path.basename(PDF_PATH),
            "source_path": PDF_PATH,
            "page_number": page_index + 1,  # Human-readable, one-based page number
            "pdf_page_index": page_index,   # Zero-based PDF index for tooling
            "page_text": page_text,
            "character_count": text_length,
            "word_count": len(re.findall(r"\S+", page_text)),
            "text_sha256": text_hash,
            "mentions_variable_content": mentions_variable,
            "mentions_statistics_content": mentions_statistics,
            "variable_like_tokens": variable_tokens[:500],
            "review_flag": review_flag,
            "extracted_at": datetime.now(timezone.utc).replace(tzinfo=None)
        })

print(f"Extracted {len(extracted_rows):,} pages from {source_page_count:,} PDF pages.")
print("Pages flagged for review:", sum(1 for r in extracted_rows if r["review_flag"] != "NONE"))

In [ ]:
# Cell 5 — Write extracted pages to Delta

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    BooleanType, ArrayType, TimestampType
)

pages_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("document_title", StringType(), True),
    StructField("codebook_version", StringType(), True),
    StructField("source_file_name", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("page_number", IntegerType(), False),
    StructField("pdf_page_index", IntegerType(), False),
    StructField("page_text", StringType(), True),
    StructField("character_count", IntegerType(), False),
    StructField("word_count", IntegerType(), False),
    StructField("text_sha256", StringType(), False),
    StructField("mentions_variable_content", BooleanType(), False),
    StructField("mentions_statistics_content", BooleanType(), False),
    StructField("variable_like_tokens", ArrayType(StringType()), True),
    StructField("review_flag", StringType(), False),
    StructField("extracted_at", TimestampType(), False)
])

pages_df = spark.createDataFrame(extracted_rows, schema=pages_schema)

if REPLACE_PAGE_TABLE:
    (pages_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(PAGES_TABLE))
else:
    (pages_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(PAGES_TABLE))

print(f"Wrote {pages_df.count():,} extracted page records to {PAGES_TABLE}")

In [ ]:
# Cell 6 — Record the processing run

from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, TimestampType

completed_at = datetime.now(timezone.utc).replace(tzinfo=None)
low_text_count = sum(1 for r in extracted_rows if r["review_flag"] != "NONE")

run_record = [{
    "run_id": run_id,
    "document_id": DOCUMENT_ID,
    "document_title": DOCUMENT_TITLE,
    "codebook_version": CODEBOOK_VERSION,
    "source_path": PDF_PATH,
    "source_file_size_bytes": int(file_size_bytes),
    "source_page_count": int(source_page_count),
    "extracted_page_count": int(len(extracted_rows)),
    "empty_or_low_text_page_count": int(low_text_count),
    "extraction_status": "SUCCESS" if len(extracted_rows) == source_page_count else "PARTIAL",
    "started_at": started_at,
    "completed_at": completed_at,
    "notes": "Page-level text extraction; review flags are heuristic and do not replace visual inspection."
}]

runs_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("document_title", StringType(), True),
    StructField("codebook_version", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("source_file_size_bytes", LongType(), False),
    StructField("source_page_count", IntegerType(), False),
    StructField("extracted_page_count", IntegerType(), False),
    StructField("empty_or_low_text_page_count", IntegerType(), False),
    StructField("extraction_status", StringType(), False),
    StructField("started_at", TimestampType(), False),
    StructField("completed_at", TimestampType(), False),
    StructField("notes", StringType(), True)
])

spark.createDataFrame(run_record, schema=runs_schema).write.format("delta").mode("append").saveAsTable(RUNS_TABLE)

print(f"Recorded processing run {run_id}")

In [ ]:
# Cell 7 — Verify counts and identify pages needing review

display(
    spark.table(PAGES_TABLE)
         .filter(f"run_id = '{run_id}'")
         .select(
             "page_number", "character_count", "word_count",
             "mentions_variable_content", "mentions_statistics_content",
             "review_flag"
         )
         .orderBy("page_number")
         .limit(50)
)

print("Current run summary:")
display(
    spark.table(RUNS_TABLE)
         .filter(f"run_id = '{run_id}'")
)

print("Pages flagged for review:")
display(
    spark.table(PAGES_TABLE)
         .filter((F.col("run_id") == run_id) & (F.col("review_flag") != "NONE"))
         .select("page_number", "character_count", "review_flag", "page_text")
         .orderBy("page_number")
)

In [ ]:
# Cell 8 — Search the extracted text (example)

# Replace the search term to test a variable, concept, or phrase.
SEARCH_TERM = "RACE"

from pyspark.sql import functions as F

display(
    spark.table(PAGES_TABLE)
         .filter(F.col("page_text").contains(SEARCH_TERM))
         .select("page_number", "review_flag", "page_text")
         .orderBy("page_number")
         .limit(20)
)

## Completion checklist

- [ ] `PDF_PATH` points to the actual Codebook PDF.
- [ ] The metadata schema can be created or already exists.
- [ ] The extracted page count matches the PDF page count.
- [ ] Pages flagged as low-text have been reviewed; scanned pages may need OCR.
- [ ] The extraction run is recorded in `hrs_document_processing_runs`.
- [ ] The original PDF remains unchanged.

## Limitations and next step

- `variable_like_tokens` is a heuristic token list, not a verified list of HRS variables.
- `mentions_variable_content` and `mentions_statistics_content` are search hints only.
- Page text extraction may not preserve complex table layouts accurately.
- This notebook does not yet create the normalized variable catalog, cohort coverage, wave coverage, or descriptive-statistics tables.

**Next notebook:** parse the extracted pages into candidate variable metadata and statistics, preserve page-level evidence, and validate the candidates against the PDF before marking them as authoritative.